# SupportGen · 02 — Baseline: FLAN-T5-small *before* fine-tuning (Model A)
To show that fine-tuning helps, we first measure the **pretrained** model on the task, using exactly
the same prompt, test tickets, decoding settings and metrics that the fine-tuned model will get later.

### Pre-training vs fine-tuning vs inference
| Stage | Who did it | Data | What changes |
|---|---|---|---|
| **Pre-training** | Google (T5) | ~750 GB of web text (C4), span-corruption objective | All weights learned from random init |
| **Instruction tuning** | Google (FLAN) | 1,800+ tasks phrased as instructions | Weights adjusted so the model follows instructions |
| **Fine-tuning** | **Us** (notebook 03) | Customer-support ticket → agent answer pairs | All weights updated on our domain |
| **Inference** | This notebook / web app | A new ticket | Nothing — weights frozen, we only generate |

This notebook does **inference only** with the off-the-shelf model ("zero-shot"): it was never
trained on our tickets.

**Runtime:** GPU (T4 is fine). ~5–10 minutes.

## 0. Setup — Google Drive & project folder
Every SupportGen notebook starts with this same cell. It mounts Google Drive so that all
notebooks share one project folder (`MyDrive/SupportGen`): notebook 01 writes the cleaned
splits there, notebooks 02–06 read them. Nothing ever needs to be uploaded from your laptop.

Set `USE_DRIVE = False` only for a throw-away run — files then live in `/content` and are lost
when the Colab runtime disconnects.

In [ ]:
# ── Setup (identical in every SupportGen notebook) ──────────────────────────
USE_DRIVE = True   # False → keep everything in /content (deleted when the runtime ends)

import os, sys
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/SupportGen"
else:
    BASE = "/content/SupportGen"
os.makedirs(BASE, exist_ok=True)
os.environ["SUPPORTGEN_BASE"] = BASE   # read by supportgen_common.py
sys.path.insert(0, BASE)
print("Project folder:", BASE)

Mounted at /content/drive
Project folder: /content/drive/MyDrive/SupportGen


In [ ]:
# Colab already ships torch, transformers, accelerate, scikit-learn, pandas, matplotlib.
# These are the extras SupportGen needs (already-installed packages are skipped).
!pip -q install evaluate rouge_score bert_score sacrebleu sentencepiece sentence-transformers

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 11.7 MB/s eta 0:00:00


In [ ]:
# Shared helpers were written to Drive by 01_dataset_preparation.ipynb
if not os.path.exists(os.path.join(BASE, "supportgen_common.py")):
    raise FileNotFoundError(
        f"supportgen_common.py not found in {BASE}. Run 01_dataset_preparation.ipynb first "
        "(with the same USE_DRIVE setting).")
import importlib, supportgen_common as sg
importlib.reload(sg)
sg.set_seed()
print("Device:", sg.get_device())

Device: cuda


## 1. Load the evaluation subset and the pretrained model
`AutoModelForSeq2SeqLM` loads T5's **encoder-decoder** architecture: the encoder reads the whole
ticket with bidirectional self-attention; the decoder writes the reply token by token, attending to
its own previous tokens (masked self-attention) and to the encoder output (cross-attention).

In [ ]:
import torch, pandas as pd, numpy as np, textwrap
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

cfg = sg.load_config()
MAX_IN, MAX_OUT = cfg["max_input_len"], cfg["max_target_len"]
MODEL_NAME = sg.BASE_MODEL          # "google/flan-t5-small"
RUN_NAME = sg.RUN_BASE

eval_df = sg.load_eval_subset()
prompts = [sg.build_prompt(s, b) for s, b in zip(eval_df.subject, eval_df.body)]
print(f"{len(eval_df)} evaluation tickets | max_input_len={MAX_IN} | max_new_tokens={MAX_OUT}")
print("\nExample prompt:\n" + prompts[0])

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(sg.get_device())
print(f"\n{MODEL_NAME}: {sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters")

500 evaluation tickets | max_input_len=160 | max_new_tokens=128

Example prompt:
Generate a professional customer support response.
Subject: Ensuring the Safety of Medical Data
Customer ticket: Requesting advice on securing medical data using Microsoft OneNote and Kaspersky Internet Security. Would like to know if these tools can protect sensitive medical information from unauthorized access. Could you provide guidance on setting up encryption and access controls to ensure the confidentiality and integrity of the medical data? Appreciate your assistance.


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


google/flan-t5-small: 77.0M parameters


## 2. Generate responses
**Decoding settings** (kept identical for every model in this project):
* **Beam search, 4 beams** — keeps the 4 most probable partial sentences at each step instead of
  greedily picking the single best next token; usually more fluent and less repetitive than greedy.
* `no_repeat_ngram_size=3` — forbids repeating any 3-word sequence (a common failure of small models).
* `max_new_tokens = max_target_len` from notebook 01.

Sampling methods (temperature, top-k, top-p) add randomness, which makes metric comparisons noisy,
so they are not used for evaluation.

In [ ]:
preds, lat = sg.generate_batch(model, tok, prompts, MAX_IN, MAX_OUT, batch_size=16)
out = eval_df[["uid", "subject", "body", "answer", "queue", "priority", "type"]].rename(columns={"answer": "reference"})
out["prediction"], out["latency_s_batched"] = preds, lat
out.to_csv(sg.outputs_path(RUN_NAME), index=False)
print("Saved", sg.outputs_path(RUN_NAME))
print(f"Mean generation time per ticket (batch of 16): {np.mean(lat):.3f}s")

Saved /content/drive/MyDrive/SupportGen/results/outputs_A_base_flan-t5-small.csv
Mean generation time per ticket (batch of 16): 0.161s


## 3. Automatic evaluation
| Metric | What it measures | Notes |
|---|---|---|
| **ROUGE-1 / ROUGE-2** | Overlap of single words / word pairs with the reference reply | Recall-oriented; standard for summarisation |
| **ROUGE-L** | Longest common subsequence | Rewards matching word order |
| **BLEU** | n-gram precision with a brevity penalty | Designed for translation; harsh for open-ended replies where many different answers are correct |
| **BERTScore** | Token similarity in contextual RoBERTa-large embeddings | Captures paraphrases ROUGE misses; rescaled version spreads scores out |
| **Semantic cosine** | Cosine similarity of sentence embeddings (MiniLM) | Overall meaning similarity |
| **Distinct-1/2** | Share of unique uni/bi-grams in outputs | Detects repetitive / generic generations |

None of these perfectly measures quality for support replies (a correct answer can be worded very
differently from the reference), which is why notebook 04 also prepares a **human evaluation**.

In [ ]:
metrics = sg.compute_gen_metrics(out.prediction.tolist(), out.reference.tolist())
metrics.update({"run_name": RUN_NAME, "model": MODEL_NAME, "fine_tuned": False,
                "mean_latency_s_batched": float(np.mean(lat)), "decoding": sg.DEFAULT_GEN_KWARGS})
sg.save_json(metrics, sg.metrics_path(RUN_NAME))
display(pd.Series(metrics).to_frame(RUN_NAME))

sg.log_experiment(RUN_NAME, model=MODEL_NAME, epochs=0, max_input_len=MAX_IN, max_target_len=MAX_OUT,
                  rouge1=metrics["rouge1"], rouge2=metrics["rouge2"], rougeL=metrics["rougeL"],
                  bleu=metrics["bleu"], bertscore_f1=metrics["bertscore_f1"],
                  semantic_cosine=metrics["semantic_cosine"],
                  notes="Zero-shot pretrained FLAN-T5-small (no domain fine-tuning)")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

... (17 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,A_base_flan-t5-small
n,500
rouge1,0.204857
rouge2,0.080117
rougeL,0.158303
bleu,1.17403
bertscore_precision,0.891949
bertscore_recall,0.843159
bertscore_f1,0.866585
bertscore_f1_rescaled,0.209509
semantic_cosine,0.579654


/content/drive/MyDrive/SupportGen/supportgen_common.py:275: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([{"run_name": run_name, **fields}])], ignore_index=True)


,run_name,model,date,epochs,learning_rate,batch_size,grad_accum,max_input_len,max_target_len,train_loss,val_loss,test_loss,rouge1,rouge2,rougeL,bleu,bertscore_f1,semantic_cosine,notes
0,A_base_flan-t5-small,google/flan-t5-small,2026-10-14 07:44,0,NaN,NaN,NaN,160,128,NaN,NaN,NaN,0.204857,0.080117,0.158303,1.17403,0.866585,0.579654,Zero-shot pretrained FLAN-T5-small (no domain ...


## 4. Look at some outputs

In [ ]:
for _, r in out.sample(5, random_state=sg.SEED).iterrows():
    print("=" * 100)
    print("SUBJECT:", r.subject)
    print("TICKET:", textwrap.shorten(r.body, 400))
    print("\nREFERENCE:", textwrap.fill(r.reference, 100))
    print("\nBASE FLAN-T5:", textwrap.fill(r.prediction, 100))

SUBJECT: Multiple Product Performance Issues Due to System Latency
TICKET: A marketing agency is encountering system performance issues that are impacting several products. The problem might be caused by network congestion. We have attempted a reboot and applied updates, but the issue remains unresolved. We would appreciate your prompt assistance to minimize operational disruptions.

REFERENCE: <name>, thank you for contacting us regarding the system performance issues affecting multiple
products for the marketing agency. We apologize for any inconvenience caused. To better understand
the problem and provide an effective solution, please share details about the nature of the
slowdowns, which specific products are affected, and any error messages encountered. We would also
like to schedule a call to discuss this further; you can reach me at <tel_num> at your convenience.
Our team is committed to resolving this issue as swiftly as possible.

BASE FLAN-T5: Customer ticket: A marketing age

## Notes for the report
Describe what you observe in the outputs above (length, relevance, tone, whether the model repeats
the ticket, etc.) using the actual generations — do not generalise beyond what you see.
The numbers are stored in `results/metrics_A_base_flan-t5-small.json` and `experiments.csv`.

➡️ **Next:** `03_train_flan_t5.ipynb`.